# CEBM MLP Training Diagnostics

This notebook inspects a continuous EBM trained on grayscale MNIST-like data.

It mirrors the BEBM MLP diagnostics: checkpoint loading, MLP spectra, saved chain samples, data-vs-chain energy histograms, PCA projections, and short HMC refreshes from selected checkpoints.

## Setup

In [1]:
import os
from pathlib import Path

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from rbms.dataset import load_dataset
from rbms.io import load_model, load_params
from rbms.utils import get_saved_updates

%load_ext autoreload
%autoreload 2

plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.family"] = "STIXGeneral"
plt.rcParams.update({"font.size": 13})

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

# Change these to your run.
MODEL_FILE = "/tmp/cebm_mnist.h5"
TRAIN_DATASET = "/tmp/mnist_continuous.h5"
TEST_DATASET = None

IMAGE_SHAPE = (28, 28)
SAMPLES_PER_ROW = 8

print("device:", device)
print("model file:", MODEL_FILE)
print("train dataset:", TRAIN_DATASET)


/Users/aidan/Documents/Research_Internship/Code/rbms/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu
model file: /tmp/cebm_mnist.h5
train dataset: /tmp/mnist_continuous.h5


## Optional: Create Continuous MNIST HDF5

Run this cell only if `TRAIN_DATASET` does not already exist. It requires `torchvision`.

In [2]:
if not Path(TRAIN_DATASET).exists():
    from torchvision.datasets import MNIST
    from torchvision import transforms

    dataset = MNIST(
        root="/tmp/mnist_data",
        train=True,
        download=True,
        transform=transforms.ToTensor(),
    )

    x = []
    y = []
    for image, label in tqdm(dataset, desc="MNIST to HDF5"):
        x.append(image.view(-1).numpy())
        y.append(label)

    x = np.asarray(x, dtype="float32")
    y = np.asarray(y, dtype="int32")

    with h5py.File(TRAIN_DATASET, "w") as f:
        f["samples"] = x
        f["labels"] = y
        f["variable_type"] = np.asarray("continuous", dtype="T")

    print("saved", TRAIN_DATASET, x.shape, x.min(), x.max())
else:
    print("dataset already exists")


100%|██████████| 9.91M/9.91M [00:03<00:00, 3.26MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 150kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.98MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]
MNIST to HDF5: 100%|██████████| 60000/60000 [00:01<00:00, 54713.72it/s]


saved /tmp/mnist_continuous.h5 (60000, 784) 0.0 1.0


## Load Data and Checkpoints

In [3]:
train_dataset, test_dataset = load_dataset(
    dataset_name=TRAIN_DATASET,
    test_dataset_name=TEST_DATASET,
    device=device,
    dtype=dtype,
)

train_visible = train_dataset.data.to(device=device, dtype=dtype)
num_visibles = train_dataset.get_num_visibles()

saved_updates = get_saved_updates(MODEL_FILE)
print("num visibles:", num_visibles)
print("first update:", int(saved_updates[0]))
print("last update: ", int(saved_updates[-1]))
print("checkpoints: ", len(saved_updates))


Reading dataset from /tmp/mnist_continuous.h5...
    Done


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/tmp/cebm_mnist.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## Helpers

In [ ]:
def selected_updates_decades(updates):
    updates = np.asarray(updates)
    max_update = int(updates[-1])
    targets = [int(updates[0])]

    decade = 100
    while decade <= max_update:
        for multiplier in range(1, 10):
            target = multiplier * decade
            if target <= max_update:
                targets.append(target)
        decade *= 10

    targets.append(max_update)
    targets = np.unique(targets)

    selected = []
    for target in targets:
        index = np.argmin(np.abs(updates - target))
        selected.append(updates[index])

    return np.unique(selected)


def selected_updates_linear(updates, num_points):
    indices = np.linspace(0, len(updates) - 1, num_points).round().astype(int)
    return np.asarray(updates)[np.unique(indices)]


def read_mlp_weights(filename, update):
    with h5py.File(filename, "r") as f:
        params_group = f[f"update_{int(update)}"]["params"]
        keys = list(params_group.keys())
        weight_keys = sorted(
            [key for key in keys if key.startswith("net.") and key.endswith(".weight")],
            key=lambda key: int(key.split(".")[1]),
        )
        weights = {
            key: torch.as_tensor(params_group[key][()], dtype=torch.float32)
            for key in weight_keys
        }
    return weights


def plot_image_rows(samples_by_update, title, samples_per_checkpoint=SAMPLES_PER_ROW):
    num_rows = len(samples_by_update)
    fig, axes = plt.subplots(
        num_rows,
        samples_per_checkpoint,
        figsize=(samples_per_checkpoint, 1.35 * num_rows),
        squeeze=False,
    )

    for row, (update, visible) in enumerate(samples_by_update.items()):
        visible = visible.detach().cpu()
        for col in range(samples_per_checkpoint):
            ax = axes[row, col]
            ax.imshow(visible[col].view(*IMAGE_SHAPE), cmap="gray_r", vmin=0, vmax=1)
            ax.axis("off")
        axes[row, 0].set_ylabel(
            f"update {update}",
            rotation=0,
            labelpad=40,
            va="center",
            fontsize=10,
        )

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def weighted_sample_rows(x, num_rows, seed=0):
    generator = torch.Generator(device=x.device).manual_seed(seed)
    idx = torch.randperm(x.shape[0], generator=generator, device=x.device)[:num_rows]
    return x[idx]


## Data Samples

In [ ]:
plot_image_rows(
    {"data": weighted_sample_rows(train_visible, SAMPLES_PER_ROW, seed=0)},
    "training data samples",
)


## MLP Weight Spectra Over Training

In [ ]:
SPECTRA_TOP_K = 10
spectra_updates = selected_updates_decades(saved_updates)

spectra_history = {
    "updates": spectra_updates,
    "W1_singular": [],
    "W1_squared_singular": [],
    "W2_singular": [],
}

for update in tqdm(spectra_updates, desc="MLP spectra"):
    weights = read_mlp_weights(MODEL_FILE, update)
    W1 = weights["net.0.weight"]
    W2 = weights["net.2.weight"]

    s1 = torch.linalg.svdvals(W1).sort(descending=True).values[:SPECTRA_TOP_K]
    s2 = torch.linalg.svdvals(W2).sort(descending=True).values[:SPECTRA_TOP_K]

    spectra_history["W1_singular"].append(s1.cpu())
    spectra_history["W1_squared_singular"].append(s1.square().cpu())
    spectra_history["W2_singular"].append(s2.cpu())

spectra_history["W1_singular"] = torch.stack(spectra_history["W1_singular"]).numpy()
spectra_history["W1_squared_singular"] = torch.stack(spectra_history["W1_squared_singular"]).numpy()
spectra_history["W2_singular"] = torch.stack(spectra_history["W2_singular"]).numpy()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
updates = spectra_history["updates"]

for k in range(SPECTRA_TOP_K):
    axes[0].plot(updates, spectra_history["W1_singular"][:, k], marker="o", ms=3)
    axes[1].plot(updates, spectra_history["W2_singular"][:, k], marker="o", ms=3)

axes[0].set_title("W1 singular values")
axes[1].set_title("W2 singular values")
for ax in axes:
    ax.set_xlabel("update")
    ax.grid(True, alpha=0.3)
    ax.set_xscale("symlog", linthresh=1)

plt.tight_layout()
plt.show()


## Saved Permanent Chains

In [ ]:
final_update = int(saved_updates[-1])
final_params, final_chains, _ = load_model(
    MODEL_FILE,
    final_update,
    device=device,
    dtype=dtype,
)

print("final chains:", final_chains["visible"].shape)
print("chain min/max:", final_chains["visible"].min().item(), final_chains["visible"].max().item())

plot_image_rows(
    {final_update: final_chains["visible"][:SAMPLES_PER_ROW].clamp(0, 1)},
    "final saved permanent chains, clipped to [0, 1] for display",
)


## Energy Histograms: Data vs Chains

In [ ]:
ENERGY_NUM_DATA = min(2048, train_visible.shape[0])
data_eval = weighted_sample_rows(train_visible, ENERGY_NUM_DATA, seed=1)
chain_eval = final_chains["visible"][: min(ENERGY_NUM_DATA, final_chains["visible"].shape[0])]

with torch.no_grad():
    data_energy = final_params.compute_energy_visibles(data_eval).detach().cpu().numpy()
    chain_energy = final_params.compute_energy_visibles(chain_eval).detach().cpu().numpy()

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.hist(data_energy, bins=50, density=True, alpha=0.55, label="data")
ax.hist(chain_energy, bins=50, density=True, alpha=0.55, label="chains")
ax.set_xlabel("energy")
ax.set_ylabel("density")
ax.set_title(f"Energy histogram at update {final_update}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("mean data energy:", data_energy.mean())
print("mean chain energy:", chain_energy.mean())


## PCA: Data vs Chains

In [ ]:
PCA_NUM_DATA = min(5000, train_visible.shape[0])
PCA_NUM_CHAINS = min(5000, final_chains["visible"].shape[0])

data_pca = weighted_sample_rows(train_visible, PCA_NUM_DATA, seed=2).detach().cpu()
chain_pca = final_chains["visible"][:PCA_NUM_CHAINS].detach().cpu()

mean = data_pca.mean(dim=0, keepdim=True)
centered = data_pca - mean
_, _, components = torch.pca_lowrank(centered, q=2, center=False)

data_proj = (data_pca - mean) @ components[:, :2]
chain_proj = (chain_pca - mean) @ components[:, :2]

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(data_proj[:, 0], data_proj[:, 1], s=5, alpha=0.25, label="data")
ax.scatter(chain_proj[:, 0], chain_proj[:, 1], s=8, alpha=0.45, label="chains")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Data PCA basis, update {final_update}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## HMC Refresh From Selected Checkpoints

In [ ]:
REFRESH_UPDATES = selected_updates_linear(saved_updates, num_points=5)
REFRESH_NUM_CHAINS = 64
REFRESH_NUM_STEPS = 25
HMC_STEP_SIZE = 1e-2
HMC_LEAPFROG_STEPS = 10
HMC_CLAMP = (0.0, 1.0)

refreshed_samples = {}
refresh_acceptance = {}

for update in tqdm(REFRESH_UPDATES, desc="HMC refresh"):
    params = load_params(MODEL_FILE, int(update), device=device, dtype=dtype)
    chains = params.init_chains(REFRESH_NUM_CHAINS)
    chains = params.sample_state(
        chains,
        n_steps=REFRESH_NUM_STEPS,
        kernel="hmc",
        kernel_params={
            "step_size": HMC_STEP_SIZE,
            "num_leapfrog_steps": HMC_LEAPFROG_STEPS,
            "clamp": HMC_CLAMP,
        },
    )
    refreshed_samples[int(update)] = chains["visible"][:SAMPLES_PER_ROW].detach().cpu()
    refresh_acceptance[int(update)] = float(chains["acceptance"].detach().cpu())

print(refresh_acceptance)


In [ ]:
plot_image_rows(
    {update: samples.clamp(0, 1) for update, samples in refreshed_samples.items()},
    "HMC refreshed samples from selected checkpoints, clipped to [0, 1]",
)
